In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# совместимый версии
!pip install torchtext=='0.18.0' torch=='2.3.0' torchdata=='0.9.0' portalocker

In [24]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torchtext import datasets
from datasets import load_dataset
from torch.utils.data import DataLoader
from torchtext.data import get_tokenizer
from torch.nn.utils.rnn import pad_sequence
from torchtext.vocab import build_vocab_from_iterator

In [66]:
train_dataset = load_dataset("go_emotions", split='train[:80%]').shuffle(seed=42)
test_dataset = load_dataset("go_emotions", split='test')

In [67]:
train_dataset.features

{'text': Value('string'),
 'labels': List(ClassLabel(names=['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'])),
 'id': Value('string')}

In [68]:
names=['admiration', 'amusement', 'anger', 'annoyance', 'approval',
       'caring', 'confusion', 'curiosity', 'desire', 'disappointment',
       'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
       'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
       'pride', 'realization', 'relief', 'remorse', 'sadness',
       'surprise', 'neutral']

In [69]:
train_dataset[0]

{'text': 'I like it I just wish he’d drop the m and apostrophe',
 'labels': [8],
 'id': 'edrmocj'}

In [70]:
train_dataset[1]

{'text': "I'm shocked, shocked, to find that redditors don't read sources and stay with the info they get from the misleading titles. ",
 'labels': [26],
 'id': 'ee32myz'}

In [71]:
tokenizer = get_tokenizer('basic_english')

In [72]:
def split_text(data):
    for i in data:
        yield tokenizer(i['text'])

In [73]:
vocab = build_vocab_from_iterator(split_text(train_dataset), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [74]:
def change_text(data):
    return [vocab[i] for i in tokenizer(data)]

In [75]:
def collate_batch(batch):
    texts = []
    labels = []

    for item in batch:
        # Текст → тензор
        text_tensor = torch.tensor(change_text(item['text']), dtype=torch.long)
        texts.append(text_tensor)

        # ←←← ИСПРАВЛЕНИЕ ЗДЕСЬ ←←←
        # Создаём one-hot вектор из 28 эмоций
        label_vector = [0.0] * len(names)
        for label_idx in item['labels']:          # item['labels'] = например [8, 15]
            if label_idx < len(names):            # защита
                label_vector[label_idx] = 1.0

        label_tensor = torch.tensor(label_vector, dtype=torch.float32)
        labels.append(label_tensor)

    # Паддинг текстов
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)

    # Стэкаем метки
    labels = torch.stack(labels)

    return texts_padded, labels

In [77]:
train_dataloader = DataLoader(
    list(train_dataset),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_batch
)

test_dataloader = DataLoader(
    list(test_dataset),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_batch
)

In [78]:
class CheckEmotionText(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=256, output_dim=28):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim,
                           batch_first=True,
                           bidirectional=True,   # оставляем
                           dropout=0.3)          # добавляем dropout

        self.lin = nn.Linear(hidden_dim * 2, output_dim)  # ← *2 из-за bidirectional!

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)

        # hidden shape: (2, batch_size, hidden_dim) — forward + backward
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)  # конкатенируем оба направления
        return self.lin(hidden)

In [79]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [80]:
model = CheckEmotionText(len(vocab)).to(device)

In [81]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [91]:
for epoch in range(500):
    model.train()
    total_loss = 0
    for texts, labels in train_dataloader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        pred = model(texts)
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Эпоха: {epoch + 1} - Потери: {round(total_loss, 2)}')

Эпоха: 1 - Потери: 2.33
Эпоха: 2 - Потери: 2.31
Эпоха: 3 - Потери: 2.42
Эпоха: 4 - Потери: 1.96
Эпоха: 5 - Потери: 1.81
Эпоха: 6 - Потери: 2.03
Эпоха: 7 - Потери: 2.35
Эпоха: 8 - Потери: 2.51
Эпоха: 9 - Потери: 2.15
Эпоха: 10 - Потери: 1.92
Эпоха: 11 - Потери: 1.87
Эпоха: 12 - Потери: 2.25
Эпоха: 13 - Потери: 2.09
Эпоха: 14 - Потери: 2.01
Эпоха: 15 - Потери: 2.02
Эпоха: 16 - Потери: 2.23
Эпоха: 17 - Потери: 2.14
Эпоха: 18 - Потери: 2.36
Эпоха: 19 - Потери: 2.46
Эпоха: 20 - Потери: 2.53
Эпоха: 21 - Потери: 1.99
Эпоха: 22 - Потери: 2.04
Эпоха: 23 - Потери: 2.03
Эпоха: 24 - Потери: 2.2
Эпоха: 25 - Потери: 2.4
Эпоха: 26 - Потери: 2.45
Эпоха: 27 - Потери: 2.26
Эпоха: 28 - Потери: 2.31
Эпоха: 29 - Потери: 2.08
Эпоха: 30 - Потери: 2.07
Эпоха: 31 - Потери: 2.04
Эпоха: 32 - Потери: 2.37
Эпоха: 33 - Потери: 2.07
Эпоха: 34 - Потери: 2.1
Эпоха: 35 - Потери: 2.11
Эпоха: 36 - Потери: 2.14
Эпоха: 37 - Потери: 2.21
Эпоха: 38 - Потери: 2.12
Эпоха: 39 - Потери: 2.25
Эпоха: 40 - Потери: 2.29
Эпоха: 41 - 

In [92]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

model.eval()
all_preds = []
all_labels = []
threshold = 0.5

with torch.no_grad():
    for x_batch, y_batch in test_dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        outputs = model(x_batch)
        probs = torch.sigmoid(outputs)
        predictions = (probs > threshold).float()

        all_preds.append(predictions.cpu().numpy())
        all_labels.append(y_batch.cpu().numpy())

all_preds  = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

exact_match = accuracy_score(all_labels, all_preds)
f1_micro    = f1_score(all_labels, all_preds, average='micro', zero_division=0)
f1_macro    = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f'Exact Match Accuracy : {exact_match * 100:.2f}%')
print(f'F1 Micro             : {f1_micro:.4f}')
print(f'F1 Macro             : {f1_macro:.4f}')

Exact Match Accuracy : 33.28%
F1 Micro             : 0.4684
F1 Macro             : 0.3397


In [93]:
from google.colab import files

torch.save(vocab, 'vocab_GoEmotions.pth')
torch.save(model.state_dict(), 'model_GoEmotions.pth')

files.download('vocab_GoEmotions.pth')
files.download('model_GoEmotions.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>